In [7]:
from collections import defaultdict
import json 
from transformers import pipeline, AutoTokenizer
import pandas as pd
import torch
import time
from tqdm import tqdm
import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"

/shared/nas2/stutia3/anaconda3/envs/eval/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/shared/nas2/stutia3/anaconda3/envs/eval/lib/python3.8/site-packages/torch/cuda/__init__.py:88: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 802: system not yet initialized (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [8]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_pipeline = pipeline("text-generation", model=model_name, torch_dtype=torch.float16, device_map="auto")

Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.97it/s]


In [1]:
import re
import numpy as np
def get_label_name(line):  
    label = line.split("[")[0]
    name = (line.split("[")[1]).split("]")[0]
    return label, name

def get_label_name(line):  
    label = line.split("[")[0]
    name = (line.split("[")[1]).split("]")[0]
    return label, name


def parse_graphs(schema_g, low_level_labels, curr_node, visited_g, curr_start):
    tos = schema_g[curr_node]
    if len(tos) == 1 and tos[0] not in schema_g["end_nodes"]:
        to = tos[0]
        if curr_start in low_level_labels:
            low_level_labels[curr_start].append(curr_node)
            visited_g[tos[0]] = 1
            parse_graphs(schema_g, low_level_labels, tos[0], visited_g, curr_start)
        else:
            low_level_labels[curr_node].append(curr_node)
            visited_g[curr_node]=1
            parse_graphs(schema_g, low_level_labels, to, visited_g, curr_node)
        
    elif len(tos) > 1 and tos[0] not in schema_g["end_node"]:
        for to in tos:
            if visited_g[to] == 0:
                visited_g[to] = 1
                parse_graphs(schema_g, low_level_labels, to, visited_g, to)

def parse_file(schema_g, nodes_g, visited_g, schema_code_file) :
    to_list = []
    fro_list = []
    with open(schema_code_file, 'r') as f:
        for line in f:
            data = line.strip()
            
            if "-->" in data:
                fro = data.split("-->")[0].strip()
                if "[" in fro:
                    fro, name = get_label_name(fro)
                    nodes_g[fro] = name
                to = data.split("-->")[1].strip()
                if "[" in to:
                    to, name = get_label_name(to)
                    nodes_g[to] = name
                to_list.append(to)
                fro_list.append(fro)
                schema_g[fro].append(to)
                visited_g[to] = 0
            else:
                if "[" in data:
                    data, name = get_label_name(data)
                    nodes_g[data] = name

    fro= ""
    for fros in fro_list:
        if fros not in to_list:
            fro = fros
    tos = []
    for to in to_list:
        if to not in fro_list:
            tos.append(to)
    return fro,  tos

def create_graph_from_edges(schema_code_file):
    schema_g = defaultdict(list)
    nodes_g = defaultdict(str)
    visited_g = defaultdict(int)
    user_node_names = dict()
    bot_node_names = dict()

    start_node, end_nodes = parse_file(schema_g, nodes_g, visited_g, schema_code_file)
    schema_g["end_node"] = end_nodes

    for node, label in nodes_g.items():
        if node[0] == "u" or node[0] == "U":
            user_node_names[node]  = label 
        elif node[0] == "b" or node[0] == "B":
            bot_node_names[node]  = label

    node_name_to_no = {}
    for no, name in nodes_g.items():
        node_name_to_no[name] = no
    return nodes_g, node_name_to_no, user_node_names, bot_node_names, schema_g

def load_conversations(conversation_file):
    conversations = []
    with open(conversation_file, 'r') as f:
        for line in f:
            conversations.append(json.loads(line.strip()))
    conversations_df = pd.DataFrame(conversations)
    convos = []
    for i, row in conversations_df.iterrows():
        convos.append(row["utterances"])

    return convos

def chat_model(context, candidates, utterance, temperature=1.0):
    generate_prompt = format_prompt(context, candidates, utterance)

    sampling = False
    if temperature > 0.0:
        sampling = True

    sequences = llm_pipeline(
        generate_prompt,
        do_sample=sampling,
        top_k=10,
        return_full_text=False,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=4,
    )

    generated_text = sequences[0]["generated_text"].strip()
    
    return generated_text
    # response = generated_text[
    #     len(generate_prompt) :
    # ]  ##Here we are removing the query that we pass onto our llm.
    # return generated_text.strip()

def format_prompt(context, candidates, utterance):
    system_prompt = """You will be provided with a dialog history and a task-oriented bot utterance. Given a set of candidate dialog bot actions, your task is to identify the most appropriate dialog action that the bot has taken in the provided utterance. You SHOULD NOT generate the entire dialog action, ONLY output the action ID. Below are some examples provided some example examples below:
    
    ### Candidate Actions: [1: Greeting and Inquiry, 2: Asks for Clarification, 3: Provides Specific Movie Info, 4: Handles Errors and Misunderstandings, 5: Provides Showtimes for Specific Movie, 6: Asks for Feedback/Satisfaction, 7: Provides Specific Movie Info from List, 8: Lists Movies at Local Theater, 9: Explains reservation limits, 10: Provides Additional Info, 11: Informs About Ticket Prices]
    
    ### Utterance: The Day of the Big Ants will be playing at 7:15 and 10:45
    ### Action: 5

    ### Utterance: Yes, the AMC near you is playing Weekend at Bernies 2, a Star Wars Marathon, and Godfather III.
    ### Action: 8

    ### Utterance: Tickets for A Quiet Place at AMC are $13.95.
    ### Action: 11

    Now, you are provided with the following dialog history and the current task-oriented bot utterance. You need to classify the given bot utterance into a dialog action from the given candidate actions. You should use the dialog history as additional context when deciding the most appropriate dialog action. You need to ONLY generate the action ID. DO NOT generate the entire action name.

    ### Context: {context}

    ### Candidate Actions: {candidates}

    ### Utterance: {utterance}

    ### Action:"""

    candidates_list = [str(i+1) + ": " + candidate for i,candidate in enumerate(candidates)]
    candidates_str = "["+", ".join(candidates_list) + "]"

    inp_prompt = system_prompt.format(context=context, candidates=candidates_str, utterance=utterance)
    messages = [
                    {"role": "user", "content": inp_prompt}
                ]
    starting_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return starting_prompt   


def pick_closest_match(query, candidates, context, attempts=3):
    
    num_retries = 0
    temperature = 0.0
    while num_retries < attempts:
        generated_text = chat_model(query, candidates, context, temperature=temperature)
        pattern = r"^\d+"
        match = re.search(pattern, generated_text)
        if match:
            return candidates[int(match.group(0))-1]
        else:
            temperature = 1.0
            num_retries +=1        

    return candidates[0]

def calculate_evaluation_score(convo_nodes, induced_schema):
    score = 0
    for i in range(len(convo_nodes) - 1):
        current_node = convo_nodes[i]
        next_node = convo_nodes[i + 1]
        
        next_user_actions = induced_schema[current_node]
        for user_action in next_user_actions:

            if next_node in induced_schema.get(user_action, []):
                # Increase the score if the transition is valid in the expected schema
                score += 1
            else:
                node_found = False
                next_to_next_user_actions = []
                bot_actions =  induced_schema.get(user_action, [])
                for bot_action in bot_actions:
                    next_to_next_user_actions += induced_schema.get(bot_action, [])
                    for next_user_action in next_to_next_user_actions:
                        if next_node in induced_schema.get(next_user_action,[]) and not node_found:
                            score += 1
                            node_found = True  
    
    # total_possible_transitions = sum(len(connected_nodes) for connected_nodes in induced_schema.values())
    return score / (len(convo_nodes)-1)  # Normalize the score to be between 0 and 1

In [3]:
dataset_files ={
    "metawoz" : { "dev" : ["conversations/Metawoz/dev", "schemas/Metawoz/dev", "results/Metawoz/dev"], "test": ["conversations/Metawoz/test", "schemas/Metawoz/test", "results/Metawoz/dev"]},
    "multiwoz" : {"test": ["conversations/Multiwoz", "schemas/Multiwoz", "results/Multiwoz"]}
}

metawoz_test_domains = ["alarm_set", "apartment_finder", "appointment_reminder","bank_bot", "bus_schedule_bot","catalogue_bot", "city_info", "edit_playlist", "event_reserve","guiness_check", "insurance", "library_request", "look_up_info", "music_suggester", "name_suggester", "pet_advice", "scam_lookup", "shopping", "ski_bot", "sports_info", "store_details", "update_calendar", "update_contact", "wedding_planner"]
metawoz_dev_domains = ["phone_plan", "order_pizza", "movie_listings", "restaurant_picker", "weather_check"]
multiwoz_domains = ["attraction", "hotel", "restaurant", "taxi", "train"]


In [5]:

def match_utterances_to_labels(dataset, conversations, user_node_names, bot_node_names, nodes_name_to_no):
    bot_candidates = {}
    for bot_node in bot_node_names.values(): 
        bname = bot_node
        if bname[0] == '"':
            bname = bname[1:-1]
        if bname.startswith("Bot:"):
            bname = bname[4:]
        bot_candidates[bname] = bot_node       

    convos_nodes = []
    for conversation in tqdm(conversations):
        convo_nodes = []
        context = []
        for i, utterance in enumerate(conversation):
            if len(context) == 3:
                context.pop(0)
            if (i % 2 == 0 and dataset == "metawoz") or (i % 2 != 0 and dataset == "multiwoz"):
                context.append("Bot: "+ utterance)
            elif (i % 2 != 0 and dataset == "metawoz") or (i % 2 == 0 and dataset == "multiwoz"):
                context.append("User: " + utterance)

            if i % 2 == 0:
                best_matches = pick_closest_match(utterance, list(bot_candidates.keys()), context)
                # print(best_matches)
                if best_matches:
                    best_match = best_matches  # Take the top-ranked match
                    node_no = nodes_name_to_no[bot_candidates[best_match]]
                    convo_nodes.append(node_no)

        convos_nodes.append(convo_nodes)
    return convos_nodes

# code_files   = ["final_schema_code/data_movie_code.txt","final_schema_code/movie_code.txt","final_schema_code/pizza_code.txt","final_schema_code/data_pizza_code.txt"]
# conversation_files = ["conversations/movie_listing.txt","conversations/movie_listing.txt","conversations/order_pizza.txt", "conversations/order_pizza.txt"]
# result_files = ["eval_numbers/data_movie.txt","eval_numbers/movie.txt","eval_numbers/pizza.txt", "eval_numbers/data_pizza.txt"]


# i = 0
# nodes_g, node_name_to_no, user_node_names, bot_node_names, schema_g = create_graph_from_edges(code_files[i])
# conversations = load_conversations(conversation_files[i])
# convos_nodes = match_utterances_to_labels("metawoz", conversations[len(conversations)-20:], user_node_names, bot_node_names, node_name_to_no)

# scores = []
# for convo_nodes in convos_nodes:
#     score = calculate_evaluation_score(convo_nodes, schema_g)
#     scores.append(score)
# print(np.mean(scores), np.max(scores), np.min(scores)) 
        

In [6]:
methods = ["llm", "data", "merged"]
for domain in metawoz_dev_domains:
    for method in methods:
        nodes_g, node_name_to_no, user_node_names, bot_node_names, schema_g = create_graph_from_edges(f'schemas/Metawoz/dev/{method}/{domain}_code.txt')
        conversations = load_conversations(f'conversations/Metawoz/dev/metawoz_test_{domain}.txt')
        convos_nodes = match_utterances_to_labels("metawoz", conversations, user_node_names, bot_node_names, node_name_to_no)

        scores = []
        for convo_nodes in convos_nodes:
            score = calculate_evaluation_score(convo_nodes, schema_g)
            scores.append(score)
        print(domain, method, np.mean(scores), np.max(scores), np.min(scores)) 


NameError: name 'defaultdict' is not defined